# **K-NN Classification**
In this notebook we will be benchmarking our algorithm performed in [the previous notebook](m11_KNN_histogram_classification.ipynb)

## **1. Import and Model Preparation**

### **1.1 Imports**

In [ ]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel

### **1.2 Environment Preparation**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from huggingface_hub import login
login()

### **1.3 Model Download**

In [ ]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m" # Flagship
#model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

## **2. Dataset**

In [ ]:
from sklearn.model_selection import train_test_split

### **2.1 Caltech 101**

In [ ]:
caltech_full = torchvision.datasets.Caltech101(
    root='./data',
    download=True
)

In [ ]:
total_images = len(caltech_full)
categories = caltech_full.categories
print(f"✅ Success! Loaded {total_images} high-resolution images.")
print(f"Total Categories: {len(categories)}\n")

In [ ]:
print("⚖️ Stratifying and splitting the dataset...")
# torchvision's Caltech101 stores the numerical labels in the '.y' attribute
all_labels = caltech_full.y
all_indices = np.arange(total_images)

# Let's use 80% of the images to build our Bag of Words, and 20% to test it
train_idx, test_idx, train_labels, test_labels = train_test_split(
    all_indices,
    all_labels,
    test_size=0.20,
    random_state=42,
    stratify=all_labels
)

## **3. Foreground Segmentation**

### **3.1 Imports and prep**

In [ ]:
import io
import urllib.request
import tarfile
from PIL import Image
from sklearn.linear_model import LogisticRegression

In [ ]:
print("Downloading Meta's Foreground Segmentation Dataset...")

IMAGES_URI = "https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_images.tar.gz"
LABELS_URI = "https://dl.fbaipublicfiles.com/dinov3/notebooks/foreground_segmentation/foreground_segmentation_labels.tar.gz"

In [ ]:
def load_tar(url, is_mask=False):
    items = []
    req = urllib.request.urlopen(url)
    tar = tarfile.open(fileobj=io.BytesIO(req.read()))

    members = sorted(tar.getmembers(), key=lambda m: m.name)
    for member in members:
        if member.isreg():
            f = tar.extractfile(member)
            if is_mask:
                # 1. Load the mask as RGBA (keeping the transparency)
                img_rgba = Image.open(f).convert("RGBA")
                # 2. Extract ONLY the Alpha channel (the actual silhouette)
                mask = img_rgba.split()[-1]
                items.append(mask)
            else:
                img = Image.open(f).convert("RGB").copy()
                items.append(img)
    return items

In [ ]:
train_images = load_tar(IMAGES_URI, is_mask=False)
train_labels = load_tar(LABELS_URI, is_mask=True)

print(f"✅ Successfully downloaded {len(train_images)} images and {len(train_labels)} masks!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(train_images[0])
axes[0].set_title("Meta Training Image", fontsize=14, fontweight='bold')
axes[0].axis('off')

# Plot the alpha channel mask using a gray colormap
axes[1].imshow(train_labels[0], cmap='gray')
axes[1].set_title("Ground Truth Mask", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### **3.2 Training Loop**

In [ ]:
IMAGE_SIZE = 512
PATCH_SIZE = 16

In [ ]:
def resize_to_patch_multiple(img, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE):
    w, h = img.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))

    new_h = h_patches * patch_size
    new_w = w_patches * patch_size

    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)

In [ ]:
X_train_list = []
y_train_list = []

In [ ]:
model.eval()
with torch.no_grad():
    for img, mask in zip(train_images, train_labels):

        img_resized = resize_to_patch_multiple(img)
        mask_resized = resize_to_patch_multiple(mask)

        w_new, h_new = img_resized.size
        grid_w = w_new // PATCH_SIZE
        grid_h = h_new // PATCH_SIZE

        num_spatial_patches = grid_w * grid_h

        inputs = processor(images=img_resized, return_tensors="pt", do_resize=False, do_center_crop=False).to(device)
        outputs = model(**inputs)

        patch_tokens = outputs.last_hidden_state[0, -num_spatial_patches:, :].cpu().numpy()

        mask_grid = mask_resized.resize((grid_w, grid_h), Image.Resampling.NEAREST)
        mask_array = np.array(mask_grid).flatten()

        binary_labels = (mask_array > 128).astype(int)

        X_train_list.append(patch_tokens)
        y_train_list.append(binary_labels)

In [ ]:
X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)
print("Training Logistic Regression model...")

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

accuracy = log_reg.score(X_train, y_train)
print(f"🎉 Model trained successfully! Training Accuracy: {accuracy * 100:.2f}%")

## **4. Feature Extraction**

In [ ]:
import os
from tqdm import tqdm

### **4.1 Function definition**

In [ ]:
def extract_and_save_features(indices, labels, dataset, split_name):
    features_list = []
    labels_list = []

    with torch.no_grad():
        for idx, label in tqdm(zip(indices, labels), total=len(indices), desc=f"Extracting {split_name.upper()}"):

            # Grab the image and FORCE it to be RGB (prevents grayscale crashes)
            img, _ = dataset[idx]
            img = img.convert("RGB")

            # Preprocess the image and send it to the GPU
            inputs = processor(images=img, return_tensors="pt").to(device)

            # Pass through the Flagship Model
            outputs = model(**inputs)

            # Sequence = [1 CLS Token] + [N Register Tokens] + [Image Patches]
            hidden_states = outputs.last_hidden_state
            patch_features = hidden_states[0, 1 + num_registers : ]

            # Move to CPU and cast to float16
            features_list.append(patch_features.cpu().to(torch.float16))
            labels_list.append(label)

    # Save the data to your hard drive using PyTorch's native save function
    filename = f"caltech101_{split_name}_flagship_features.pt"
    save_dict = {
        "features": features_list,
        "labels": labels_list
    }

    torch.save(save_dict, filename)
    print(f"✅ Successfully saved {len(features_list)} images to {filename}\n")

### **4.2 Extracting Train and Test**

In [ ]:
extract_and_save_features(train_idx, train_labels, caltech_full, "train")

In [ ]:
extract_and_save_features(test_idx, test_labels, caltech_full, "test")

### **3.3 Loading the data**

In [ ]:
train_data = torch.load("caltech101_train_flagship_features.pt", weights_only=False)
test_data = torch.load("caltech101_test_flagship_features.pt", weights_only=False)

In [ ]:
caltech_train_features = [tensor.to(torch.float32).numpy() for tensor in train_data["features"]]
caltech_train_labels = train_data["labels"]

caltech_test_features = [tensor.to(torch.float32).numpy() for tensor in test_data["features"]]
caltech_test_labels = test_data["labels"]

## **5. Clustering**
We will be clustering via `FAISS` instead of standard `sklearn`

### **5.1 Imports**

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import time

### **5.2 Wrapper Class**

In [ ]:
class FaissKMeansWrapper:
    def __init__(self, k, d, niter=20):
        self.k = k
        self.d = d
        self.kmeans = faiss.Kmeans(d, k, niter=niter, verbose=False, gpu=False)
        self.index = None

    def fit(self, x):
        """Trains the K-Means model to find the cluster centers."""
        self.kmeans.train(x)
        self.index = faiss.IndexFlatL2(self.d)
        self.index.add(self.kmeans.centroids)

    def predict(self, x):
        """Behaves exactly like scikit-learn"""
        if x.dtype != np.float32:
            x = x.astype(np.float32)
        distances, nearest_centroid_indices = self.index.search(x, 1)
        return nearest_centroid_indices.flatten()

### **5.3 Running the clustering**

In [ ]:
all_train_patches = np.vstack(caltech_train_features).astype(np.float32)

num_total_patches, dim = all_train_patches.shape
print(f"✅ Matrix built! Total Patches: {num_total_patches} | Dimensions: {dim}")

In [ ]:
vocab_sizes = [16, 64, 256, 1024]
caltech_kmeans_models = {}

In [ ]:
for k in vocab_sizes:
    print(f"\n🧩 Training FAISS K-Means for Vocabulary Size K = {k}...")
    start_time = time.time()

    # Initialize and train our custom FAISS model
    faiss_model = FaissKMeansWrapper(k=k, d=dim)
    faiss_model.fit(all_train_patches)

    # Save it to our global dictionary so the rest of your pipeline can use it!
    caltech_kmeans_models[k] = faiss_model

    elapsed = time.time() - start_time
    print(f"   -> Completed in {elapsed:.1f} seconds!")

### **5.4 Visualisation of clusters**

In [ ]:
k_vis = 64
kmeans_model = caltech_kmeans_models[k_vis]

## **6. Data modification**

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
from collections import Counter
import matplotlib.pyplot as plt

### **6.1 Function definition**

In [ ]:
def generate_raw_counts(features_list, model, vocab_size):
    raw_counts = []
    for patches in features_list:
        labels = model.predict(patches)
        counts = np.bincount(labels, minlength=vocab_size)
        raw_counts.append(counts)
    return np.array(raw_counts)

In [ ]:
vocab_sizes = [16, 64, 256, 1024]
knn_neighbors = [1, 3, 7, 9, 11, 25]

### **6.2 Histogram Generation**

In [ ]:
caltech_raw_train_hists = {}
caltech_raw_test_hists  = {}

In [ ]:
for k in vocab_sizes:
    print(f"   -> Generating histograms for K={k}...")
    kmeans_model = caltech_kmeans_models[k]

    train_counts = generate_raw_counts(caltech_train_features, kmeans_model, k)
    test_counts = generate_raw_counts(caltech_test_features, kmeans_model, k)

    caltech_raw_train_hists[k] = train_counts
    caltech_raw_test_hists[k] = test_counts

### **6.3 Histogram Plotting**

In [ ]:
This is a brilliant idea! What you are asking for is exactly what computer vision researchers call the "Class Signature." By averaging the histograms of every single image within a specific category, you can see the fundamental "DNA" of that class. You will immediately notice that a Leopard has a massive spike on the visual words representing "spots" and "fur," while an Airplanes signature will spike heavily on "blue sky" and "smooth metal."

To make this completely bulletproof, the function below will:

    Normalize the histograms first (converting raw counts into percentages) so every image contributes equally.

    Average them together.

    Wrap dynamically so if you ask for 10 classes, it will perfectly generate two rows of 5, never exceeding your maximum of 6 columns!

Here is your dynamic Class Signature plotter. Drop this into your notebook:
Python

import numpy as np
import matplotlib.pyplot as plt

print("📊 Initializing Class Signature Plotter...")

def plot_average_class_histograms(class_names, k=64, max_cols=6):
    """
    Plots the average visual word distribution (signature) for requested classes.

    Args:
        class_names (list of str): The exact names of the categories to plot.
        k (int): The vocabulary size to pull from your cached histograms.
        max_cols (int): Maximum number of histograms per row.
    """

    categories = caltech_full.categories

    target_labels = []
    for name in class_names:
        clean_name = name.lower()
        if clean_name in categories:
            target_labels.append((clean_name, categories.index(clean_name)))
        else:
            print(f"⚠️ Warning: Class '{name}' not found in Caltech-101 dataset.")

    num_plots = len(target_labels)
    if num_plots == 0:
        print("❌ No valid classes found to plot.")
        return

    # Calculate the dynamic grid layout
    cols = min(max_cols, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    fig.suptitle(f"Average Class Signatures (Vocab K={k})", fontsize=20, fontweight='bold', y=1.05)

    # Flatten axes array to make looping easy, even if it's just 1 row
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    # Grab our pre-cached raw counts
    train_raw = caltech_raw_train_hists[k]
    caltech_train_labels_array = np.array(caltech_train_labels)

    for i, (name, label_idx) in enumerate(target_labels):
        img_indices = np.where(caltech_train_labels_array == label_idx)[0]

        class_hists = train_raw[img_indices]

        # Convert raw counts to percentages (L1 Normalization)
        class_hists_norm = class_hists / (np.sum(class_hists, axis=1, keepdims=True) + 1e-10)

        avg_hist = np.mean(class_hists_norm, axis=0)

        ax = axes[i]
        ax.bar(range(k), avg_hist, color='royalblue', edgecolor='black', linewidth=0.5, alpha=0.8)

        ax.set_title(f"{name.title()} ({len(img_indices)} imgs)", fontsize=14, fontweight='bold')
        ax.set_xlabel("Visual Word ID", fontsize=10)
        ax.set_ylabel("Average Frequency (%)", fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.5)


    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
classes_to_compare = [
    "leopard", "airplanes", "faces", "motorbikes",
    "flamingo", "watch", "brain", "sunflower"
]

In [ ]:
plot_average_class_histograms(classes_to_compare, k=64, max_cols=5)

## **7. Chi-Square - K-NN**

### **7.1 Matrix Init**

In [ ]:
chisq_results_matrix = np.zeros((len(vocab_sizes), len(knn_neighbors)))

### **7.2 Evaluation**

In [ ]:
for i, vocab_k in enumerate(vocab_sizes):
    print(f"\n🧩 Evaluating Chi-Square for K = {vocab_k}")
    start_time = time.time()

    # LOAD FROM CACHE! No more .predict() waiting!
    train_raw = caltech_raw_train_hists[vocab_k]
    test_raw = caltech_raw_test_hists[vocab_k]

    # L1 Normalize (Convert counts to percentages) for Chi-Square
    # We add a tiny epsilon (1e-10) to prevent dividing by zero
    train_hists = train_raw / (np.sum(train_raw, axis=1, keepdims=True) + 1e-10)
    test_hists = test_raw / (np.sum(test_raw, axis=1, keepdims=True) + 1e-10)

    # Vectorized Chi-Square Distance Matrix Calculation
    num_test = len(test_hists)
    num_train = len(train_hists)
    dist_matrix = np.zeros((num_test, num_train))

    for t_idx in range(num_test):
        test_vec = test_hists[t_idx]

        # MATH MAGIC: Broadcasts the calculation across all 7,300 train images at once!
        numerator = (train_hists - test_vec) ** 2
        denominator = train_hists + test_vec + 1e-10
        dist_matrix[t_idx, :] = 0.5 * np.sum(numerator / denominator, axis=1)

    # Standard K-NN Loop
    for j, nn in enumerate(knn_neighbors):
        correct = 0
        for test_idx in range(num_test):
            distances = dist_matrix[test_idx]
            nearest_indices = np.argsort(distances)[:nn]
            nearest_labels = [caltech_train_labels[idx] for idx in nearest_indices]

            pred_label = Counter(nearest_labels).most_common(1)[0][0]
            if pred_label == caltech_test_labels[test_idx]:
                correct += 1

        accuracy = (correct / len(caltech_test_labels)) * 100
        chisq_results_matrix[i, j] = accuracy
        print(f"   -> {nn}-NN Accuracy: {accuracy:.1f}%")

    elapsed = time.time() - start_time
    print(f"   ⏱️ Completed K={vocab_k} in {elapsed:.1f} seconds")

### **7.3 Plotting**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.imshow(chisq_results_matrix, cmap='Greens', vmin=0, vmax=100)

cbar = fig.colorbar(cax)
cbar.set_label('Chi-Square Classification Accuracy (%)', fontsize=14, fontweight='bold')

ax.set_xticks(np.arange(len(knn_neighbors)))
ax.set_yticks(np.arange(len(vocab_sizes)))
ax.set_xticklabels([f"{k}-NN" for k in knn_neighbors], fontsize=12, fontweight='bold')
ax.set_yticklabels([f"Vocab K={v}" for v in vocab_sizes], fontsize=12, fontweight='bold')
ax.set_xlabel("Number of Neighbors (K-NN)", fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel("Semantic Vocabulary Size", fontsize=14, fontweight='bold', labelpad=10)

for i in range(len(vocab_sizes)):
    for j in range(len(knn_neighbors)):
        text_color = "white" if chisq_results_matrix[i, j] > 50 else "black"
        ax.text(j, i, f"{chisq_results_matrix[i, j]:.1f}%",
                ha="center", va="center", color=text_color, fontsize=14, fontweight='bold')

ax.set_title("Flagship Baseline: Chi-Square K-NN\n(Full Caltech-101)", fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## **8. TD-IDF**

### **8.1 Init**

In [ ]:
knn_neighbors = [1, 3, 7, 9, 11, 25]
vocab_sizes = [64, 256, 1024]
tfidf_faiss_results_matrix = np.zeros((len(vocab_sizes), len(knn_neighbors)))

### **8.2 Running the loop**

In [ ]:
for i, vocab_k in enumerate(vocab_sizes):
    print(f"\n🧩 Processing Vocabulary Size K = {vocab_k}...")
    start_time = time.time()

    # LOAD FROM CACHE! Instant access to raw histograms
    train_raw = caltech_raw_train_hists[vocab_k]
    test_raw = caltech_raw_test_hists[vocab_k]

    # Apply TF-IDF Weighting
    # This automatically penalizes the "Spam" background clusters
    tfidf_transformer = TfidfTransformer()
    train_tfidf = tfidf_transformer.fit_transform(train_raw).toarray().astype(np.float32)
    test_tfidf = tfidf_transformer.transform(test_raw).toarray().astype(np.float32)

    # The FAISS Cosine Trick (L2 Normalize + Inner Product)
    faiss.normalize_L2(train_tfidf)
    faiss.normalize_L2(test_tfidf)

    # Initialize the FAISS Inner Product Index (which now equals Cosine Similarity!)
    index = faiss.IndexFlatIP(vocab_k)
    index.add(train_tfidf)

    # Test all K-NN values instantly
    for j, nn in enumerate(knn_neighbors):
        # Search FAISS. Because it's Cosine, HIGHER scores are better!
        distances, nearest_indices = index.search(test_tfidf, nn)

        correct = 0
        for test_idx in range(len(caltech_test_labels)):
            # Filter out any weird -1 indices (just a FAISS safety net)
            valid_indices = [idx for idx in nearest_indices[test_idx] if idx != -1]
            nearest_lbls = [caltech_train_labels[idx] for idx in valid_indices]

            if nearest_lbls:
                pred_label = Counter(nearest_lbls).most_common(1)[0][0]
                if pred_label == caltech_test_labels[test_idx]:
                    correct += 1

        accuracy = (correct / len(caltech_test_labels)) * 100
        tfidf_faiss_results_matrix[i, j] = accuracy
        print(f"   -> {nn}-NN Accuracy: {accuracy:.1f}%")

    elapsed = time.time() - start_time
    print(f"   ⏱️ Completed K={vocab_k} TF-IDF Search in {elapsed:.1f} seconds")

### **8.3 Plotting**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Using a distinct 'Blues' colormap so you can easily compare it against the Green Chi-Square
cax = ax.imshow(tfidf_faiss_results_matrix, cmap='Blues', vmin=0, vmax=100)

cbar = fig.colorbar(cax)
cbar.set_label('TF-IDF + Cosine Classification Accuracy (%)', fontsize=14, fontweight='bold')

ax.set_xticks(np.arange(len(knn_neighbors)))
ax.set_yticks(np.arange(len(vocab_sizes)))
ax.set_xticklabels([f"{k}-NN" for k in knn_neighbors], fontsize=12, fontweight='bold')
ax.set_yticklabels([f"Vocab K={v}" for v in vocab_sizes], fontsize=12, fontweight='bold')
ax.set_xlabel("Number of Neighbors (FAISS K-NN)", fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel("Semantic Vocabulary Size", fontsize=14, fontweight='bold', labelpad=10)

for i in range(len(vocab_sizes)):
    for j in range(len(knn_neighbors)):
        text_color = "white" if tfidf_faiss_results_matrix[i, j] > 50 else "black"
        ax.text(j, i, f"{tfidf_faiss_results_matrix[i, j]:.1f}%",
                ha="center", va="center", color=text_color, fontsize=14, fontweight='bold')

ax.set_title("Flagship Upgrade: TF-IDF + FAISS Cosine\n(Full Uncropped Caltech-101)", fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## **9. Evaluation**

In [ ]:
import random

### **9.1 Function Definition**

In [ ]:
def plot_diagnostic_grid(error_list, method_name, num_examples=3):
    if len(error_list) < num_examples:
        num_examples = len(error_list)

    sample_errors = random.sample(error_list, num_examples)

    fig, axes = plt.subplots(num_examples, 4, figsize=(20, 4 * num_examples))
    fig.suptitle(f"Diagnostic: Why {method_name} Failed (K={k_eval})", fontsize=20, fontweight='bold', y=1.02)
    discrete_cmap = plt.get_cmap('turbo', k_eval)

    if num_examples == 1:
        axes = [axes]

    for row, (t_idx, train_idx) in enumerate(sample_errors):
        # Grab actual image indices
        orig_test_idx = test_idx[t_idx]
        orig_train_idx = train_idx[train_idx] # Note: local var masking, fixed below

        # Real indices from the global datasets
        global_test_idx = test_idx[t_idx]
        global_train_idx = train_idx[train_idx]

        # Get Images
        test_img, _ = caltech_full[global_test_idx]
        test_img = test_img.convert("RGB").resize((224, 224))

        train_img, _ = caltech_full[global_train_idx]
        train_img = train_img.convert("RGB").resize((224, 224))

        # Get Labels
        true_label = caltech_full.categories[caltech_test_labels[t_idx]].title()
        wrong_label = caltech_full.categories[caltech_train_labels[train_idx]].title()

        test_patches = caltech_test_features[t_idx]
        test_grid = kmeans_model.predict(test_patches).reshape((14, 14))

        train_patches = caltech_train_features[train_idx]
        train_grid = kmeans_model.predict(train_patches).reshape((14, 14))

        axes[row][0].imshow(test_img)
        axes[row][0].set_title(f"Test Image (True: {true_label})", fontsize=14, fontweight='bold')
        axes[row][0].axis('off')

        axes[row][1].imshow(test_img)
        axes[row][1].imshow(test_grid, cmap=discrete_cmap, alpha=0.55, extent=[0, 224, 224, 0], vmin=-0.5, vmax=k_eval-0.5)
        axes[row][1].set_title(f"{true_label} Clusters", fontsize=14, color='darkblue')
        axes[row][1].axis('off')

        axes[row][2].imshow(train_img)
        axes[row][2].set_title(f"Closest Train (Pred: {wrong_label})", fontsize=14, color='darkred', fontweight='bold')
        axes[row][2].axis('off')

        axes[row][3].imshow(train_img)
        axes[row][3].imshow(train_grid, cmap=discrete_cmap, alpha=0.55, extent=[0, 224, 224, 0], vmin=-0.5, vmax=k_eval-0.5)
        axes[row][3].set_title(f"{wrong_label} Clusters", fontsize=14, color='darkblue')
        axes[row][3].axis('off')

    plt.tight_layout()
    plt.show()

### **9.2 Evaluation**

In [ ]:
chi_errors = []

In [ ]:
k_eval = 64
kmeans_model = caltech_kmeans_models[k_eval]

train_raw = caltech_raw_train_hists[k_eval]
test_raw = caltech_raw_test_hists[k_eval]


train_hists_l1 = train_raw / (np.sum(train_raw, axis=1, keepdims=True) + 1e-10)
test_hists_l1 = test_raw / (np.sum(test_raw, axis=1, keepdims=True) + 1e-10)

In [ ]:
# We'll just calculate the distances for the first few hundred test images to save time,
for t_idx in range(min(500, len(test_hists_l1))):
    test_vec = test_hists_l1[t_idx]
    numerator = (train_hists_l1 - test_vec) ** 2
    denominator = train_hists_l1 + test_vec + 1e-10
    distances = 0.5 * np.sum(numerator / denominator, axis=1)

    closest_train_idx = np.argmin(distances)

    if caltech_test_labels[t_idx] != caltech_train_labels[closest_train_idx]:
        chi_errors.append((t_idx, closest_train_idx))

In [ ]:
# ==========================================
# FIND TF-IDF MISTAKES (1-NN)
# ==========================================
tfidf_transformer = TfidfTransformer()
train_tfidf = tfidf_transformer.fit_transform(train_raw).toarray().astype(np.float32)
test_tfidf = tfidf_transformer.transform(test_raw).toarray().astype(np.float32)

faiss.normalize_L2(train_tfidf)
faiss.normalize_L2(test_tfidf)

index = faiss.IndexFlatIP(k_eval)
index.add(train_tfidf)
distances, nearest_indices = index.search(test_tfidf, 1) # 1-NN search

tfidf_errors = []
for t_idx in range(min(500, len(test_tfidf))):
    closest_train_idx = nearest_indices[t_idx][0]
    if caltech_test_labels[t_idx] != caltech_train_labels[closest_train_idx]:
        tfidf_errors.append((t_idx, closest_train_idx))

### **9.3 Plotting**

In [ ]:
plot_diagnostic_grid(chi_errors, "Chi-Square", num_examples=3)
plot_diagnostic_grid(tfidf_errors, "TF-IDF + Cosine", num_examples=3)

# **Cropping The Images**

## **10. Initialisation**

### **10.1 Function definition**

In [ ]:
knn_neighbors = [1, 3, 7, 9, 11, 25]
vocab_sizes = [64, 256, 1024]
cropped_tfidf_results_matrix = np.zeros((len(vocab_sizes), len(knn_neighbors)))

In [ ]:
def generate_cropped_raw_counts(features_list, kmeans_model, vocab_size, log_reg_model):
    """
    Uses the trained Logistic Regression to filter out background patches
    BEFORE passing them to K-Means.
    """
    raw_counts = []

    for patches in features_list:
        # patches shape: (Num_Patches, 4096)
        binary_mask = log_reg_model.predict(patches)

        foreground_patches = patches[binary_mask == 1]

        # SAFETY NET: If the LR model hallucinates and deletes the entire image,
        # fallback to using all patches so the math doesn't crash!
        if len(foreground_patches) == 0:
            foreground_patches = patches

        # Assign the surviving foreground patches to visual words
        labels = kmeans_model.predict(foreground_patches)
        counts = np.bincount(labels, minlength=vocab_size)
        raw_counts.append(counts)

    return np.array(raw_counts)

### **10.2 Histogram Generation**

In [ ]:
caltech_cropped_train_hists = {}
caltech_cropped_test_hists = {}

for k in vocab_sizes:
    print(f"   -> Generating Cropped Histograms for K={k}...")
    kmeans_model = caltech_kmeans_models[k]

    train_counts = generate_cropped_raw_counts(caltech_train_features, kmeans_model, k, log_reg)
    test_counts = generate_cropped_raw_counts(caltech_test_features, kmeans_model, k, log_reg)

    caltech_cropped_train_hists[k] = train_counts
    caltech_cropped_test_hists[k] = test_counts

print("✅ Cropped Histograms cached!\n")

## **11. Chi-Square and FAISS L2**

### **11.1 Evaluation**

In [ ]:
cropped_chisq_results = np.zeros((len(vocab_sizes), len(knn_neighbors)))
cropped_l2_results = np.zeros((len(vocab_sizes), len(knn_neighbors)))

for i, vocab_k in enumerate(vocab_sizes):
    print(f"\n🧩 Evaluating Baselines for Cropped K = {vocab_k}")
    start_time = time.time()

    # LOAD FROM CACHE
    train_raw = caltech_cropped_train_hists[vocab_k]
    test_raw = caltech_cropped_test_hists[vocab_k]

    # L1 Normalize (Convert counts to percentages)
    train_hists_l1 = train_raw / (np.sum(train_raw, axis=1, keepdims=True) + 1e-10)
    test_hists_l1 = test_raw / (np.sum(test_raw, axis=1, keepdims=True) + 1e-10)

    num_test = len(test_hists_l1)
    num_train = len(train_hists_l1)
    dist_matrix_chisq = np.zeros((num_test, num_train))

    for t_idx in range(num_test):
        test_vec = test_hists_l1[t_idx]
        numerator = (train_hists_l1 - test_vec) ** 2
        denominator = train_hists_l1 + test_vec + 1e-10
        dist_matrix_chisq[t_idx, :] = 0.5 * np.sum(numerator / denominator, axis=1)

    # The Math Hack: Square root the normalized histograms for Hellinger Distance
    train_hellinger = np.sqrt(train_hists_l1).astype(np.float32)
    test_hellinger = np.sqrt(test_hists_l1).astype(np.float32)

    index_l2 = faiss.IndexFlatL2(vocab_k)
    index_l2.add(train_hellinger)

    # --- EVALUATE K-NN FOR BOTH ---
    for j, nn in enumerate(knn_neighbors):
        # Chi-Square Eval
        correct_chi = 0
        for test_idx in range(num_test):
            nearest_indices_chi = np.argsort(dist_matrix_chisq[test_idx])[:nn]
            nearest_labels_chi = [caltech_train_labels[idx] for idx in nearest_indices_chi]
            pred_chi = Counter(nearest_labels_chi).most_common(1)[0][0]
            if pred_chi == caltech_test_labels[test_idx]:
                correct_chi += 1
        cropped_chisq_results[i, j] = (correct_chi / num_test) * 100

        # FAISS L2 Eval (LOWER distance is better in L2)
        distances_l2, nearest_indices_l2 = index_l2.search(test_hellinger, nn)
        correct_l2 = 0
        for test_idx in range(num_test):
            valid_indices_l2 = [idx for idx in nearest_indices_l2[test_idx] if idx != -1]
            nearest_labels_l2 = [caltech_train_labels[idx] for idx in valid_indices_l2]
            if nearest_labels_l2:
                pred_l2 = Counter(nearest_labels_l2).most_common(1)[0][0]
                if pred_l2 == caltech_test_labels[test_idx]:
                    correct_l2 += 1
        cropped_l2_results[i, j] = (correct_l2 / num_test) * 100

    elapsed = time.time() - start_time
    print(f"   ⏱️ Completed K={vocab_k} in {elapsed:.1f} seconds")

### **11.2 Plotting**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("Cropped Image Baselines: Chi-Square vs. FAISS L2 (Hellinger)", fontsize=22, fontweight='bold', y=1.05)

# Plot Cropped Chi-Square
cax1 = axes[0].imshow(cropped_chisq_results, cmap='Purples', vmin=0, vmax=100)
cbar1 = fig.colorbar(cax1, ax=axes[0], fraction=0.046, pad=0.04)
cbar1.set_label('Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title("Cropped: Chi-Square K-NN", fontsize=16, fontweight='bold', pad=15)

# Plot Cropped FAISS L2
cax2 = axes[1].imshow(cropped_l2_results, cmap='Reds', vmin=0, vmax=100)
cbar2 = fig.colorbar(cax2, ax=axes[1], fraction=0.046, pad=0.04)
cbar2.set_label('Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_title("Cropped: FAISS L2 (Hellinger) K-NN", fontsize=16, fontweight='bold', pad=15)

for ax, matrix in zip(axes, [cropped_chisq_results, cropped_l2_results]):
    ax.set_xticks(np.arange(len(knn_neighbors)))
    ax.set_yticks(np.arange(len(vocab_sizes)))
    ax.set_xticklabels([f"{k}-NN" for k in knn_neighbors], fontsize=12, fontweight='bold')
    ax.set_yticklabels([f"Vocab K={v}" for v in vocab_sizes], fontsize=12, fontweight='bold')
    ax.set_xlabel("Number of Neighbors", fontsize=14, fontweight='bold', labelpad=10)
    ax.set_ylabel("Semantic Vocabulary Size", fontsize=14, fontweight='bold', labelpad=10)

    for i in range(len(vocab_sizes)):
        for j in range(len(knn_neighbors)):
            text_color = "white" if matrix[i, j] > 50 else "black"
            ax.text(j, i, f"{matrix[i, j]:.1f}%",
                    ha="center", va="center", color=text_color, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## **12. TD-IDF**

In [ ]:
cropped_tfidf_results = np.zeros((len(vocab_sizes), len(knn_neighbors)))

### **12.1 Running the thing**

In [ ]:
for i, vocab_k in enumerate(vocab_sizes):
    print(f"\n🧩 Weighting and Searching Cropped K = {vocab_k}...")
    start_time = time.time()

    train_raw = caltech_cropped_train_hists[vocab_k]
    test_raw = caltech_cropped_test_hists[vocab_k]

    # This automatically finds the "Spam" words in the cropped animals and penalizes them
    tfidf_transformer = TfidfTransformer()
    train_tfidf = tfidf_transformer.fit_transform(train_raw).toarray().astype(np.float32)
    test_tfidf = tfidf_transformer.transform(test_raw).toarray().astype(np.float32)

    # FAISS COSINE TRICK
    # L2 Normalize the vectors so Inner Product becomes Cosine Similarity
    faiss.normalize_L2(train_tfidf)
    faiss.normalize_L2(test_tfidf)

    index_cosine = faiss.IndexFlatIP(vocab_k)
    index_cosine.add(train_tfidf)

    # EVALUATE K-NN
    for j, nn in enumerate(knn_neighbors):
        # Higher score is better for Cosine (Inner Product)
        distances, nearest_indices = index_cosine.search(test_tfidf, nn)

        correct = 0
        for test_idx in range(len(caltech_test_labels)):
            valid_indices = [idx for idx in nearest_indices[test_idx] if idx != -1]
            nearest_labels = [caltech_train_labels[idx] for idx in valid_indices]

            if nearest_labels:
                pred = Counter(nearest_labels).most_common(1)[0][0]
                if pred == caltech_test_labels[test_idx]:
                    correct += 1

        accuracy = (correct / len(caltech_test_labels)) * 100
        cropped_tfidf_results[i, j] = accuracy
        print(f"   -> {nn}-NN Accuracy: {accuracy:.1f}%")

    elapsed = time.time() - start_time
    print(f"   ⏱️ Completed K={vocab_k} TF-IDF in {elapsed:.1f} seconds")

### **12.2 Plotting**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cax = ax.imshow(cropped_tfidf_results, cmap='Oranges', vmin=0, vmax=100)

cbar = fig.colorbar(cax)
cbar.set_label('Cropped TF-IDF + Cosine Accuracy (%)', fontsize=14, fontweight='bold')

ax.set_xticks(np.arange(len(knn_neighbors)))
ax.set_yticks(np.arange(len(vocab_sizes)))
ax.set_xticklabels([f"{k}-NN" for k in knn_neighbors], fontsize=12, fontweight='bold')
ax.set_yticklabels([f"Vocab K={v}" for v in vocab_sizes], fontsize=12, fontweight='bold')
ax.set_xlabel("Number of Neighbors (FAISS K-NN)", fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel("Semantic Vocabulary Size", fontsize=14, fontweight='bold', labelpad=10)

for i in range(len(vocab_sizes)):
    for j in range(len(knn_neighbors)):
        text_color = "white" if cropped_tfidf_results[i, j] > 50 else "black"
        ax.text(j, i, f"{cropped_tfidf_results[i, j]:.1f}%",
                ha="center", va="center", color=text_color, fontsize=14, fontweight='bold')

ax.set_title("The Final Metric: Cropped TF-IDF + Cosine", fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()